In [ ]:
# ============================================================
# FINAL visualization -- -- for comparing quantum vs classical models---
#   note --- make sure the required data file are in place  ----- or update the folders path appropriately
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

SAVE_DIR = "paper_figures"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Paths (MATCH YOUR ZIP STRUCTURE)
# ------------------------------------------------------------
CLASSICAL_HIST_PATH = "/kaggle/input/classicaltokenizer/classical_histories.npy"
QUANTUM_HIST_PATH   = "/kaggle/input/quantum-model/qlstm_histories.npy"
QUANTUM_CM_PATH     = "/kaggle/input/quantum-model/confusion_matrix.npy"
QUANTUM_METRICS_CSV = "/kaggle/input/quantum-model/qlstm_metrics.csv"

# ------------------------------------------------------------
# Load artifacts
# ------------------------------------------------------------
classical_hist = np.load(CLASSICAL_HIST_PATH, allow_pickle=True)
quantum_hist   = np.load(QUANTUM_HIST_PATH, allow_pickle=True)
quantum_cm     = np.load(QUANTUM_CM_PATH)
quantum_metrics = np.loadtxt(
    QUANTUM_METRICS_CSV,
    delimiter=",",
    skiprows=1
)

# ------------------------------------------------------------
# Helper: mean curve with natural stopping
# ------------------------------------------------------------
def mean_curve(histories, key):
    max_len = max(len(h[key]) for h in histories)
    padded = [h[key] + [np.nan] * (max_len - len(h[key])) for h in histories]
    return np.nanmean(np.array(padded), axis=0)

# ============================================================
# FIGURE 1: VALIDATION ACCURACY (CLEAN)
# ============================================================
classical_val_acc = mean_curve(classical_hist, "val_accuracy")
quantum_val_acc   = mean_curve(quantum_hist, "val_accuracy")

epochs_classical = np.arange(1, len(classical_val_acc) + 1)
epochs_quantum   = np.arange(1, len(quantum_val_acc) + 1)

plt.figure(figsize=(7,5))
plt.plot(epochs_classical, classical_val_acc, linewidth=2, label="Classical BiLSTM")
plt.plot(epochs_quantum, quantum_val_acc, linewidth=2, linestyle="--",
         label="Quantum BiLSTM + QLSTM")

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy Comparison")

plt.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, -0.25),
    ncol=2,
    frameon=False
)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig_validation_accuracy_clean.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 2: VALIDATION LOSS (CLEAN)
# ============================================================
classical_val_loss = mean_curve(classical_hist, "val_loss")
quantum_val_loss   = mean_curve(quantum_hist, "val_loss")

epochs_classical = np.arange(1, len(classical_val_loss) + 1)
epochs_quantum   = np.arange(1, len(quantum_val_loss) + 1)

plt.figure(figsize=(7,5))
plt.plot(epochs_classical, classical_val_loss, linewidth=2, label="Classical BiLSTM")
plt.plot(epochs_quantum, quantum_val_loss, linewidth=2, linestyle="--",
         label="Quantum BiLSTM + QLSTM")

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss Comparison")

plt.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.25),
    ncol=2,
    frameon=False
)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig_validation_loss_clean.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 3: CONFUSION MATRIX (EXACT COUNTS)
# ============================================================
plt.figure(figsize=(4.8,4.4))
sns.heatmap(
    quantum_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Quantum BiLSTM + QLSTM Confusion Matrix")

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig_confusion_matrix_quantum.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 4: TEST METRICS BAR CHART (MEAN)
# ============================================================
metric_names = ["Accuracy", "Precision", "Recall", "F1-score"]
metric_means = quantum_metrics[:, :4].mean(axis=0)

plt.figure(figsize=(6,4))
plt.bar(metric_names, metric_means)
plt.ylim(0.75, 0.85)
plt.ylabel("Score")
plt.title("Quantum Model Test Performance")

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig_test_metrics_quantum.png", dpi=300)
plt.show()

print("[INFO] CLEAN, PAPER-READY FIGURES GENERATED")
print("[INFO] Saved in:", SAVE_DIR)


In [ ]:
# ============================================================
#  CONFUSION MATRIX and METRICS TABLE
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

SAVE_DIR = "paper_figures"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Paths (MATCH YOUR ZIP STRUCTURE)
# ------------------------------------------------------------
CLASSICAL_METRICS_CSV = "/kaggle/input/classicaltokenizer/classical_metrics.csv"
QUANTUM_METRICS_CSV   = "/kaggle/input/quantum-model/qlstm_metrics.csv"

# Optional (quantum exists, classical usually does not)
CLASSICAL_CM_PATH = "/kaggle/input/classicaltokenizer/confusion_matrix.npy"
QUANTUM_CM_PATH   = "/kaggle/input/quantum-model/confusion_matrix.npy"

# ============================================================
# 1️⃣ CONFUSION MATRICES
# ============================================================

def plot_confusion(cm, title, filename):
    plt.figure(figsize=(4.8,4.4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Negative", "Positive"],
        yticklabels=["Negative", "Positive"]
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

# --- Classical Confusion Matrix (if available) ---
if os.path.exists(CLASSICAL_CM_PATH):
    classical_cm = np.load(CLASSICAL_CM_PATH)
    plot_confusion(
        classical_cm,
        "Classical BiLSTM Confusion Matrix",
        f"{SAVE_DIR}/fig_confusion_matrix_classical.png"
    )
else:
    print("[INFO] Classical confusion matrix not stored.")
    print("       Test performance is reported via aggregated metrics.")

# --- Quantum Confusion Matrix ---
quantum_cm = np.load(QUANTUM_CM_PATH)
plot_confusion(
    quantum_cm,
    "Quantum BiLSTM + QLSTM Confusion Matrix",
    f"{SAVE_DIR}/fig_confusion_matrix_quantum.png"
)

# ============================================================
#  UNIFIED METRICS TABLE (CLASSICAL vs QUANTUM)
# ============================================================

# Load metrics
classical_df = pd.read_csv(CLASSICAL_METRICS_CSV)
quantum_df   = pd.read_csv(QUANTUM_METRICS_CSV)

# Compute mean ± std
def summarize(df):
    return {
        "Accuracy": f"{df['accuracy'].mean():.4f} ± {df['accuracy'].std():.4f}",
        "Precision": f"{df['precision'].mean():.4f} ± {df['precision'].std():.4f}",
        "Recall": f"{df['recall'].mean():.4f} ± {df['recall'].std():.4f}",
        "F1-score": f"{df['f1'].mean():.4f} ± {df['f1'].std():.4f}"
    }

summary_table = pd.DataFrame.from_dict(
    {
        "Classical BiLSTM": summarize(classical_df),
        "Quantum BiLSTM + QLSTM": summarize(quantum_df)
    },
    orient="index"
)

# Save table
summary_table.to_csv(f"{SAVE_DIR}/table_metrics_comparison.csv")

# Display
print("\n=== CLASSICAL vs QUANTUM TEST PERFORMANCE ===\n")
display(summary_table)

print("\n[INFO] Metrics table saved to:")
print(f"       {SAVE_DIR}/table_metrics_comparison.csv")

print("\n[INFO] All reviewer-requested visualizations completed.")
